In [ ]:
import numpy as np
from scipy import integrate
from scipy.optimize import minimize
from scipy.stats import chi2 as chi2_dist
import matplotlib.pyplot as plt

from astropy import units as u
from astropy import constants as const
from astropy.cosmology import FlatLambdaCDM

# ── Fiducial cosmology ────────────────────────────────────────────────────────
H0_fid      = 70.0 * u.km / u.s / u.Mpc
Omega_b_fid = 0.049
Omega_m_fid = 0.3
cosmo_fid   = FlatLambdaCDM(H0=H0_fid, Om0=Omega_m_fid, Ob0=Omega_b_fid)

f_IGM = 0.83   # fraction of baryons in the IGM (fixed)

# ── Data ─────────────────────────────────────────────────────────────────────
data   = np.loadtxt('data/frb_sample_000100.txt')
z_obs  = data[:, 0]   # observed redshifts
DM_obs = data[:, 1]   # observed DMs [pc/cm³]
N_frb  = len(z_obs)
print(f'Loaded {N_frb} FRBs,  z ∈ [{z_obs.min():.3f}, {z_obs.max():.3f}]')



# ── Signal model ──────────────────────────────────────────────────────────────
def DM_cosmic(z, H0, Omega_b, Omega_m, n_points=1024):
    """Mean cosmic DM [pc/cm³] via numerical integration of the line-of-sight integral."""
    cosmo     = FlatLambdaCDM(H0=H0, Om0=Omega_m, Ob0=Omega_b)
    prefactor = (3 * const.c * f_IGM * Omega_b * H0) / (8 * np.pi * const.G * const.m_p)
    prefactor = prefactor.to(u.pc / u.cm**3)

    is_scalar = np.isscalar(z)
    z_arr     = np.atleast_1d(z)
    z_prime   = np.linspace(0, np.max(z_arr), n_points)
    integrand = (1 + z_prime) / cosmo.efunc(z_prime)
    cumul     = np.concatenate(([0.0], integrate.cumulative_simpson(integrand, x=z_prime)))

    result = prefactor * np.interp(z_arr, z_prime, cumul)
    return result[0] if is_scalar else result


# ── Per-FRB variance model ────────────────────────────────────────────────────
SIGMA_HOST = 100.0  # host-galaxy DM uncertainty [pc/cm³]

def sigma_host_sq(z):
    """Host-galaxy DM variance: σ_host²/(1+z)²  [pc/cm³]²"""
    return SIGMA_HOST**2 / (1.0 + z)**2

def sigma_cosmic_sq(z):
    """Cosmic line-of-sight DM variance (polynomial fit)  [pc/cm³]²"""
    return -1.186e4 * z**2 + 4.449e4 * z + 381.3

def sigma_sq(z):
    """Total per-FRB DM variance = host + cosmic  [pc/cm³]²"""
    return sigma_host_sq(z) + sigma_cosmic_sq(z)

Loaded 100 FRBs,  z ∈ [0.102, 1.557]


# 4.2 MLE for Fast Radio Burst Cosmology

Fast radio bursts (FRBs) are millisecond radio transients whose dispersion measure (DM) accumulates along the line of sight. For a localised FRB at redshift $z$, the average cosmic contribution is

$$\overline{\mathrm{DM}}_{\mathrm{cosmic}}(z;\boldsymbol{\theta}) = \frac{3cH_0\Omega_{\mathrm{b}} f_{\mathrm{IGM}}}{8\pi G m_p} \int_0^z \frac{1+z'}{E(z';\Omega_{\mathrm{m}})}\,\mathrm{d}z',$$

where $E(z) = \sqrt{\Omega_{\mathrm{m}}(1+z)^3 + (1-\Omega_{\mathrm{m}})}$ for a flat $\Lambda\mathrm{CDM}$ universe, $f_{\mathrm{IGM}} = 0.83$ is fixed, and the free parameters are $\boldsymbol{\theta} = (H_0, \Omega_{\mathrm{b}}, \Omega_{\mathrm{m}})$. We fit these parameters to a real catalogue of $N = 100$ localised FRBs and work through the MLE workflow of Fig. 19 step by step.

**Pre-implemented for you:**
 - `DM_cosmic(z, H0, Omega_b, Omega_m)` — evaluates $\overline{\mathrm{DM}}_{\mathrm{cosmic}}$ via numerical integration
 - `sigma_host_sq(z)`, `sigma_cosmic_sq(z)`, `sigma_sq(z)` — the per-FRB variance and its two components
 - Data loading cell (reads `data/frb_sample_000100.txt` into arrays `z_obs`, `DM_obs`)

---

## Step 1 — Model: $f(\mathrm{DM}\,|\,\boldsymbol{\theta})$

Each observed DM is the sum of the cosmic signal and noise:

$$\mathrm{DM}_j^{\mathrm{obs}} = \overline{\mathrm{DM}}_{\mathrm{cosmic}}(z_j;\boldsymbol{\theta}) + n_j, \qquad n_j \sim \mathcal{N}\!\left(0,\,\sigma_j^2(z_j)\right),$$

where the total per-FRB variance $\sigma_j^2(z_j)$ is provided by the pre-implemented function `sigma_sq(z)`. The FRBs are assumed independent.

**a.** *(pre-implemented)* Inspect the noise budget plot and identify which variance term dominates at low and high redshift (stay below redshift three).

---

## Step 2 — Log-likelihood: $L(\boldsymbol{\theta})$

**b.** Write down the log-likelihood for the observed catalogue of $N$ FRBs,

$$L(\boldsymbol{\theta}) = \sum_{j=1}^N \ln f\!\left(\mathrm{DM}_j^{\mathrm{obs}}\,\big|\,\boldsymbol{\theta}\right).$$

Plot $L(H_0)$ for the loaded data with $\Omega_{\mathrm{b}}$ and $\Omega_{\mathrm{m}}$ held fixed at their fiducial values. Inspect the plot: where is the maximum, and how wide is the peak?

---

## Step 3 — MLE: $\hat{\boldsymbol{\theta}}_\mathrm{MLE}$

**c.** Implement `neg_log_like(theta)` for $\boldsymbol{\theta} = (H_0, \Omega_{\mathrm{b}}, \Omega_{\mathrm{m}})$ and find $\hat{\boldsymbol{\theta}}_\mathrm{MLE}$ by minimising it with `scipy.optimize.minimize`. Use the fiducial values as the starting point. Report the MLE and compare to the fiducial cosmology.

**d.** Compute the goodness-of-fit: evaluate $\chi^2_\mathrm{min} = -2L(\hat{\boldsymbol{\theta}}_\mathrm{MLE})$, the reduced $\chi^2_\mathrm{red} = \chi^2_\mathrm{min}/\nu$ with $\nu = N - 3$, and the $p$-value (Eq. 4.37). Is the fit acceptable?

---

## Step 4 — Fisher matrix: $\mathbf{F}$ and $\mathbf{F}^{-1}$

The Fisher information matrix evaluated at the MLE is (Eq. 4.19):

$$F_{\alpha\beta} = -\partial_\alpha\partial_\beta L\big|_{\hat{\boldsymbol{\theta}}_\mathrm{MLE}},$$

which equals the Hessian of $-L$ at the best-fit point. Since the MLE saturates the Cramér–Rao bound asymptotically, $\mathbf{F}^{-1}$ gives the covariance of $\hat{\boldsymbol{\theta}}_\mathrm{MLE}$ (Eq. 4.27).

**e. Analytical degeneracy.** Compute $\partial\overline{\mathrm{DM}}/\partial H_0$ and $\partial\overline{\mathrm{DM}}/\partial\Omega_{\mathrm{b}}$ analytically and show they are proportional. What does this imply for $\mathbf{F}$? Break the degeneracy by adding a Gaussian BBN information $\Omega_{\mathrm{b}} = 0.049 \pm 0.003$ as a diagonal term:

$$F_{\alpha\beta}^{\mathrm{total}} = F_{\alpha\beta} + \mathrm{diag}\!\left(0,\, \sigma_{\Omega_\mathrm{b}}^{-2},\, 0\right).$$

**f.** Compute $\mathbf{F}^{\mathrm{total}}$ numerically as the Hessian of $-L$ at $\hat{\boldsymbol{\theta}}_\mathrm{MLE}$ (use central finite differences), add the BBN prior, and invert to obtain the covariance matrix $\mathbf{C} = (\mathbf{F}^{\mathrm{total}})^{-1}$. Report the marginal uncertainties $\sigma_\alpha = \sqrt{C_{\alpha\alpha}}$ (Eq. 4.49).

**g.** Plot the correlation matrix of $\mathbf{C}$. Which parameter pair is most degenerate?

---

## Step 5 — Confidence regions

**h.** Using the confidence region criterion $L(\hat{\boldsymbol{\theta}}) - L(\boldsymbol{\theta}) \leq \tfrac{1}{2}\chi^2_{d,1-\alpha}$ (Eq. 4.44), produce a corner plot of the $68\%$ and $95\%$ Fisher ellipses for all three parameter pairs, with the marginal 1D Gaussians on the diagonal. Use the $\chi^2_{d=2}$ thresholds from Table 1 ($\Delta\chi^2 = 2.30$ and $5.99$).

---

## Step 6 — Visualise and validate

**i.** Overplot $\overline{\mathrm{DM}}_{\mathrm{cosmic}}(z;\hat{\boldsymbol{\theta}}_\mathrm{MLE})$ on the data $(z_j, \mathrm{DM}_j^{\mathrm{obs}})$ with error bars $\sigma_j(z_j)$. Does the best-fit curve pass through the data within the uncertainties?

---

## Step 7 — Scaling with survey size

The data directory contains four catalogues with $N = 100,\, 1\,000,\, 10\,000,\, 100\,000$ FRBs:

```
data/frb_sample_000100.txt
data/frb_sample_001000.txt
data/frb_sample_010000.txt
data/frb_sample_100000.txt
```

**j.** For each catalogue, compute $\mathbf{F}^{\mathrm{total}}$ (with BBN prior) at the fiducial cosmology and read off $\sigma_{H_0} = \sqrt{(\mathbf{F}^{-1})_{H_0 H_0}}$. Plot $\sigma_{H_0}$ as a function of $N$ on a log–log scale. What do you see?